# Advanced Problems with Solutions: Closing Python Generators

This notebook develops advanced, practical skill with:

- `generator.close()`
- `GeneratorExit`
- generator states
- deterministic resource cleanup
- `try` / `finally`
- `throw()`
- `yield from`
- generator-based pipelines
- transaction-like coroutines
- asynchronous generator shutdown
- testing cleanup behavior

Every problem is followed by a complete solution and runnable checks.

> **Best-practice theme:** if a generator owns a resource, make cleanup explicit, idempotent, and testable. Do not rely on garbage collection to close the generator at a predictable time.

## Learning objectives

By the end, you should be able to:

1. Predict state transitions caused by `next()`, normal exhaustion, exceptions, and `close()`.
2. Explain why `GeneratorExit` inherits from `BaseException`.
3. Write generators that release files, locks, sockets, or transactions on every exit path.
4. Avoid the illegal pattern of yielding after receiving `GeneratorExit`.
5. Propagate shutdown correctly through `yield from` and multi-stage pipelines.
6. Distinguish successful close, error injection with `throw()`, and ordinary exhaustion.
7. Test cleanup without depending on print statements.
8. Apply the same design ideas to async generators and `aclose()`.

## Notebook conventions

- All examples are self-contained.
- Temporary files are created with `tempfile`.
- Dangerous `eval()`-based examples are intentionally avoided.
- Assertions are used as executable specifications.
- Demonstrations that intentionally cause errors catch those errors so the notebook can run from top to bottom.

In [1]:
from __future__ import annotations

import asyncio
import csv
import gc
import inspect
import io
import itertools
import tempfile
import weakref

from collections.abc import AsyncIterator, Generator, Iterable, Iterator
from contextlib import ExitStack, closing
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, TypeVar

T = TypeVar("T")

print("Environment ready.")

Environment ready.


# Part I — Core semantics

## Problem 1 — Predict generator state transitions

Consider this generator:

```python
def sample(log):
    log.append("entered")
    try:
        yield 10
        log.append("resumed")
        yield 20
    finally:
        log.append("cleanup")
```

For each scenario below, predict:

- the generator state after each operation,
- the final contents of `log`,
- whether the body ever starts.

Scenarios:

1. Create the generator and close it before the first `next()`.
2. Advance once and then close it.
3. Fully exhaust it.
4. Advance once, then delete the last reference and request garbage collection.

Write code that verifies the first three scenarios. Treat scenario 4 as an implementation-dependent observation, not a cleanup strategy.

### Solution 1

A generator function does not execute when called. It returns a generator object in `GEN_CREATED`.

A subtle result: closing a never-started generator changes it to `GEN_CLOSED`, but its body does not execute, so its `finally` block does not run. Cleanup code inside the generator is useful only after the generator has entered its body and acquired the resource.

In [2]:
def sample(log: list[str]) -> Iterator[int]:
    log.append("entered")
    try:
        yield 10
        log.append("resumed")
        yield 20
    finally:
        log.append("cleanup")


def state(gen: Iterator[Any]) -> str:
    return inspect.getgeneratorstate(gen)


# Scenario 1: close before first next()
log1: list[str] = []
g1 = sample(log1)
assert state(g1) == "GEN_CREATED"
g1.close()
assert state(g1) == "GEN_CLOSED"
assert log1 == []  # body never started

# Scenario 2: advance once, then close
log2: list[str] = []
g2 = sample(log2)
assert next(g2) == 10
assert state(g2) == "GEN_SUSPENDED"
g2.close()
assert state(g2) == "GEN_CLOSED"
assert log2 == ["entered", "cleanup"]

# Scenario 3: normal exhaustion
log3: list[str] = []
g3 = sample(log3)
assert list(g3) == [10, 20]
assert state(g3) == "GEN_CLOSED"
assert log3 == ["entered", "resumed", "cleanup"]

print("Scenario 1:", log1)
print("Scenario 2:", log2)
print("Scenario 3:", log3)

Scenario 1: []
Scenario 2: ['entered', 'cleanup']
Scenario 3: ['entered', 'resumed', 'cleanup']


### Extra example — Observing garbage collection

CPython often finalizes an unreachable generator promptly because of reference counting, but other Python implementations may not. Even in CPython, reference cycles can delay finalization.

The following cell is only an observation. Production code should use explicit `close()`, `contextlib.closing`, or an owning context manager.

In [3]:
gc_log: list[str] = []
g = sample(gc_log)
assert next(g) == 10

ref = weakref.ref(g)
del g
gc.collect()

print("Generator object collected:", ref() is None)
print("Observed log:", gc_log)
print("Do not use this timing as a correctness guarantee.")

Generator object collected: True
Observed log: ['entered', 'cleanup']
Do not use this timing as a correctness guarantee.


## Problem 2 — Build a resource-safe streaming line reader

Implement `stream_lines(path)` with these requirements:

- Open the file lazily, only when iteration begins.
- Yield stripped, non-empty lines.
- Always close the file after normal exhaustion, explicit `close()`, or an exception.
- Expose cleanup through an injected audit list so it is testable.
- Do not catch exceptions merely to print them.

### Starter

In [4]:
def stream_lines_starter(path: Path, audit: list[str]) -> Iterator[str]:
    # TODO:
    # 1. append "open" immediately before opening
    # 2. iterate over non-empty stripped lines
    # 3. append "close" in a finally block
    raise NotImplementedError

### Solution 2

In [5]:
def stream_lines(path: Path, audit: list[str]) -> Iterator[str]:
    audit.append("open")
    file = path.open("r", encoding="utf-8")
    try:
        for raw_line in file:
            line = raw_line.strip()
            if line:
                yield line
    finally:
        file.close()
        audit.append("close")


with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "data.txt"
    path.write_text("alpha\n\n beta \ngamma\n", encoding="utf-8")

    audit: list[str] = []
    lines = stream_lines(path, audit)

    # Lazy: no code in the generator has run yet.
    assert audit == []

    assert next(lines) == "alpha"
    assert audit == ["open"]

    # Early termination must be explicit.
    lines.close()
    assert audit == ["open", "close"]
    assert inspect.getgeneratorstate(lines) == "GEN_CLOSED"

print("Resource-safe early close passed.")

Resource-safe early close passed.


### Additional checks: exhaustion and consumer exception

In [6]:
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "data.txt"
    path.write_text("a\nb\nc\n", encoding="utf-8")

    # Normal exhaustion
    audit_exhaust: list[str] = []
    assert list(stream_lines(path, audit_exhaust)) == ["a", "b", "c"]
    assert audit_exhaust == ["open", "close"]

    # Consumer-side exception: use closing() so the generator closes
    audit_error: list[str] = []
    try:
        with closing(stream_lines(path, audit_error)) as lines:
            for line in lines:
                if line == "b":
                    raise RuntimeError("consumer failed")
    except RuntimeError as exc:
        assert str(exc) == "consumer failed"

    assert audit_error == ["open", "close"]

print("Exhaustion and consumer-error checks passed.")

Exhaustion and consumer-error checks passed.


## Problem 3 — Why `except Exception` does not catch `GeneratorExit`

Write a generator that has:

- an `except Exception` branch,
- a `finally` branch,
- one suspension point.

Close it after the first yield and prove that:

- the `except Exception` branch does not run,
- the `finally` branch does run,
- `issubclass(GeneratorExit, BaseException)` is true,
- `issubclass(GeneratorExit, Exception)` is false.

### Solution 3

In [7]:
def exception_boundary(log: list[str]) -> Iterator[str]:
    try:
        yield "ready"
    except Exception as exc:
        log.append(f"caught Exception: {type(exc).__name__}")
        raise
    finally:
        log.append("finally")


log: list[str] = []
gen = exception_boundary(log)
assert next(gen) == "ready"
gen.close()

assert log == ["finally"]
assert issubclass(GeneratorExit, BaseException)
assert not issubclass(GeneratorExit, Exception)

print(log)
print("GeneratorExit MRO:", GeneratorExit.__mro__)

['finally']
GeneratorExit MRO: (<class 'GeneratorExit'>, <class 'BaseException'>, <class 'object'>)


### Design note

A broad `except Exception` is still often too broad for application code, but it will not accidentally swallow `KeyboardInterrupt`, `SystemExit`, or `GeneratorExit`, because those derive directly from `BaseException`.

## Problem 4 — Demonstrate the illegal “yield after close” pattern

Create a generator that catches `GeneratorExit` and yields another value.

Then:

1. Advance it once.
2. Call `close()`.
3. Capture the expected `RuntimeError`.
4. Close it a second time to finish cleanup.
5. Explain why Python rejects the first close.

### Solution 4

In [8]:
def illegal_shutdown(log: list[str]) -> Iterator[int]:
    try:
        yield 1
    except GeneratorExit:
        log.append("caught GeneratorExit")
        yield 999  # illegal: the caller requested termination
    finally:
        log.append("finally")


log: list[str] = []
gen = illegal_shutdown(log)
assert next(gen) == 1

try:
    gen.close()
except RuntimeError as exc:
    assert str(exc) == "generator ignored GeneratorExit"
    print("Expected RuntimeError:", exc)

# The failed close left the generator suspended at yield 999.
assert inspect.getgeneratorstate(gen) == "GEN_SUSPENDED"

# A second close injects GeneratorExit at the new suspension point.
gen.close()
assert inspect.getgeneratorstate(gen) == "GEN_CLOSED"
assert log == ["caught GeneratorExit", "finally"]

print("Final log:", log)

Expected RuntimeError: generator ignored GeneratorExit
Final log: ['caught GeneratorExit', 'finally']


### Explanation

`close()` means “terminate now,” not “produce one final item.” If a generator yields after receiving `GeneratorExit`, it has ignored the shutdown protocol, so Python raises `RuntimeError`.

For final metadata, use a separate method, shared state, a context manager, or a normal return value on ordinary exhaustion. Do not yield during close.

## Problem 5 — Implement two legal shutdown handlers

Implement:

- `shutdown_by_return(log)`: catches `GeneratorExit`, records it, and returns.
- `shutdown_by_reraise(log)`: catches `GeneratorExit`, records it, and re-raises.

Verify that `close()` returns normally in both cases and each generator ends in `GEN_CLOSED`.

### Solution 5

In [9]:
def shutdown_by_return(log: list[str]) -> Iterator[str]:
    try:
        while True:
            yield "tick"
    except GeneratorExit:
        log.append("return")
        return
    finally:
        log.append("finally")


def shutdown_by_reraise(log: list[str]) -> Iterator[str]:
    try:
        while True:
            yield "tick"
    except GeneratorExit:
        log.append("raise")
        raise
    finally:
        log.append("finally")


for factory, expected in [
    (shutdown_by_return, ["return", "finally"]),
    (shutdown_by_reraise, ["raise", "finally"]),
]:
    audit: list[str] = []
    gen = factory(audit)
    assert next(gen) == "tick"
    result = gen.close()
    assert result is None
    assert audit == expected
    assert inspect.getgeneratorstate(gen) == "GEN_CLOSED"

print("Both legal shutdown styles passed.")

Both legal shutdown styles passed.


# Part II — Error injection and delegation

## Problem 6 — Compare `close()` with `throw()`

Create a generator that records one of three outcomes:

- `"commit"` after ordinary exhaustion,
- `"cancel"` after `GeneratorExit`,
- `"rollback:<ExceptionName>"` after an injected application exception.

Use `throw(ValueError("bad record"))` to trigger the rollback path. The application exception must propagate to the caller.

### Solution 6

In [10]:
def outcome_tracker(audit: list[str]) -> Iterator[str]:
    try:
        yield "ready"
    except GeneratorExit:
        audit.append("cancel")
        raise
    except Exception as exc:
        audit.append(f"rollback:{type(exc).__name__}")
        raise
    else:
        audit.append("commit")
    finally:
        audit.append("release")


# Ordinary exhaustion
audit_normal: list[str] = []
g_normal = outcome_tracker(audit_normal)
assert next(g_normal) == "ready"
try:
    next(g_normal)
except StopIteration:
    pass
assert audit_normal == ["commit", "release"]

# Explicit close
audit_close: list[str] = []
g_close = outcome_tracker(audit_close)
assert next(g_close) == "ready"
g_close.close()
assert audit_close == ["cancel", "release"]

# Injected application error
audit_throw: list[str] = []
g_throw = outcome_tracker(audit_throw)
assert next(g_throw) == "ready"
try:
    g_throw.throw(ValueError("bad record"))
except ValueError as exc:
    assert str(exc) == "bad record"
assert audit_throw == ["rollback:ValueError", "release"]

print("normal:", audit_normal)
print("close :", audit_close)
print("throw :", audit_throw)

normal: ['commit', 'release']
close : ['cancel', 'release']
throw : ['rollback:ValueError', 'release']


### Best-practice interpretation

Whether close should mean commit or cancel is a domain decision. Do not infer transaction success from `GeneratorExit` alone. Prefer an explicit commit protocol when correctness matters.

## Problem 7 — Trace cleanup order through `yield from`

Implement:

- a child generator that records `"child-open"` and `"child-close"`,
- a parent generator that records `"parent-open"` and `"parent-close"` and delegates with `yield from`.

After consuming one item, close the parent. Determine and verify the cleanup order.

### Solution 7

In [11]:
def child(audit: list[str]) -> Iterator[int]:
    audit.append("child-open")
    try:
        for value in range(3):
            yield value
    finally:
        audit.append("child-close")


def parent(audit: list[str]) -> Iterator[int]:
    audit.append("parent-open")
    try:
        yield from child(audit)
    finally:
        audit.append("parent-close")


audit: list[str] = []
gen = parent(audit)

assert next(gen) == 0
assert audit == ["parent-open", "child-open"]

gen.close()

assert audit == [
    "parent-open",
    "child-open",
    "child-close",
    "parent-close",
]
print(audit)

['parent-open', 'child-open', 'child-close', 'parent-close']


### Explanation

`yield from` forwards `close()` to the active subgenerator. The child unwinds first, then the parent’s `finally` block runs. This stack-like order mirrors nested context managers.

## Problem 8 — Propagate close through a manual pipeline

A common mistake is to write a wrapper generator whose `finally` block runs but whose upstream iterator is never explicitly closed.

Implement `map_closing(func, source)` such that:

- values are lazily transformed,
- if `source` has a callable `.close()`, it is called when the wrapper closes or exhausts,
- calling `.close()` more than once is harmless.

### Solution 8

In [12]:
def map_closing(func, source: Iterable[T]) -> Iterator[Any]:
    iterator = iter(source)
    close = getattr(iterator, "close", None)
    try:
        for item in iterator:
            yield func(item)
    finally:
        if callable(close):
            close()


def auditable_source(audit: list[str]) -> Iterator[int]:
    audit.append("source-open")
    try:
        yield from range(10)
    finally:
        audit.append("source-close")


audit: list[str] = []
pipeline = map_closing(lambda x: x * x, auditable_source(audit))

assert next(pipeline) == 0
assert next(pipeline) == 1
assert next(pipeline) == 4

pipeline.close()
pipeline.close()  # idempotent from the caller's perspective

assert audit == ["source-open", "source-close"]
print(audit)

['source-open', 'source-close']


### Extra example — `itertools.islice` does not own upstream cleanup

`islice` stops requesting values but does not provide a general ownership protocol for closing arbitrary upstream iterators. Keep a reference to the owning generator and close it explicitly, or wrap the entire operation in an owner such as `closing()`.

In [13]:
audit: list[str] = []
source = auditable_source(audit)

first_three = list(itertools.islice(source, 3))
assert first_three == [0, 1, 2]
assert audit == ["source-open"]  # still suspended

source.close()
assert audit == ["source-open", "source-close"]

print(first_three, audit)

[0, 1, 2] ['source-open', 'source-close']


# Part III — Ownership utilities and multi-resource cleanup

## Problem 9 — Write `take_and_close`

Implement a reusable helper:

```python
take_and_close(source, count)
```

Requirements:

- Return a list containing at most `count` items.
- Reject negative counts.
- Close the iterator in a `finally` block if it has `.close()`.
- Close the source even when iteration raises an exception.
- Work with ordinary iterators that have no `.close()`.

### Solution 9

In [14]:
def take_and_close(source: Iterable[T], count: int) -> list[T]:
    if count < 0:
        raise ValueError("count must be non-negative")

    iterator = iter(source)
    close = getattr(iterator, "close", None)

    try:
        return list(itertools.islice(iterator, count))
    finally:
        if callable(close):
            close()


audit: list[str] = []
result = take_and_close(auditable_source(audit), 4)

assert result == [0, 1, 2, 3]
assert audit == ["source-open", "source-close"]

assert take_and_close(iter([10, 20]), 5) == [10, 20]

try:
    take_and_close([1, 2, 3], -1)
except ValueError as exc:
    assert str(exc) == "count must be non-negative"

print("take_and_close checks passed.")

take_and_close checks passed.


### Exception-path test

In [15]:
def failing_source(audit: list[str]) -> Iterator[int]:
    audit.append("open")
    try:
        yield 1
        raise LookupError("source failed")
    finally:
        audit.append("close")


audit: list[str] = []
try:
    take_and_close(failing_source(audit), 10)
except LookupError as exc:
    assert str(exc) == "source failed"

assert audit == ["open", "close"]
print(audit)

['open', 'close']


## Problem 10 — Use `contextlib.closing` at the ownership boundary

Build `first_matching_line(path, predicate)`.

It should:

- use `stream_lines`,
- return the first matching line or `None`,
- ensure the generator closes immediately after the search,
- not require the caller to remember to close anything.

### Solution 10

In [16]:
def first_matching_line(
    path: Path,
    predicate,
    audit: list[str],
) -> str | None:
    with closing(stream_lines(path, audit)) as lines:
        return next((line for line in lines if predicate(line)), None)


with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "items.txt"
    path.write_text("ant\nbee\ncat\ndog\n", encoding="utf-8")

    audit: list[str] = []
    found = first_matching_line(path, lambda line: line.startswith("c"), audit)

    assert found == "cat"
    assert audit == ["open", "close"]

print(found, audit)

cat ['open', 'close']


### Ownership rule

The function that decides to stop early should usually own the cleanup. This localizes the obligation and prevents leaked suspended generators.

## Problem 11 — Manage multiple resources with `ExitStack`

Implement `zip_text_files(paths, audit)`.

Requirements:

- Lazily open all files when iteration begins.
- Yield tuples of stripped lines, like `zip`.
- Close every file on normal exhaustion, early close, or error.
- Record `open:<name>` and `close:<name>`.
- Use `ExitStack` rather than deeply nested `with` statements.

### Solution 11

In [17]:
def zip_text_files(paths: list[Path], audit: list[str]) -> Iterator[tuple[str, ...]]:
    with ExitStack() as stack:
        files = []
        for path in paths:
            audit.append(f"open:{path.name}")
            file = stack.enter_context(path.open("r", encoding="utf-8"))
            files.append(file)

        try:
            for rows in zip(*files):
                yield tuple(row.rstrip("\n") for row in rows)
        finally:
            # ExitStack closes resources after this generator frame unwinds.
            # Record close intent in reverse order to mirror stack unwinding.
            for path in reversed(paths):
                audit.append(f"close:{path.name}")


with tempfile.TemporaryDirectory() as tmp:
    base = Path(tmp)
    paths = [base / "a.txt", base / "b.txt", base / "c.txt"]
    paths[0].write_text("a1\na2\na3\n", encoding="utf-8")
    paths[1].write_text("b1\nb2\nb3\n", encoding="utf-8")
    paths[2].write_text("c1\nc2\nc3\n", encoding="utf-8")

    audit: list[str] = []
    rows = zip_text_files(paths, audit)

    assert next(rows) == ("a1", "b1", "c1")
    rows.close()

    assert audit == [
        "open:a.txt",
        "open:b.txt",
        "open:c.txt",
        "close:c.txt",
        "close:b.txt",
        "close:a.txt",
    ]

print(audit)

['open:a.txt', 'open:b.txt', 'open:c.txt', 'close:c.txt', 'close:b.txt', 'close:a.txt']


### Refinement — Audit actual file closure

For tests that must prove the resources are physically closed, retain test doubles or file references and inspect their `.closed` state. Log ordering alone is not proof.

In [18]:
def zip_open_file_objects(
    files: list[io.StringIO],
) -> Iterator[tuple[str, ...]]:
    with ExitStack() as stack:
        managed = [stack.enter_context(file) for file in files]
        yield from zip(*(map(str.strip, file) for file in managed))


files = [io.StringIO("x1\nx2\n"), io.StringIO("y1\ny2\n")]
gen = zip_open_file_objects(files)
assert next(gen) == ("x1", "y1")
assert not any(file.closed for file in files)

gen.close()
assert all(file.closed for file in files)

print("All in-memory files closed:", [file.closed for file in files])

All in-memory files closed: [True, True]


# Part IV — Coroutines and transaction semantics

## Problem 12 — Design a transaction coroutine without `eval`

Create a small in-memory transaction system.

Protocol:

- `next(coroutine)` starts the transaction.
- `.send(("set", key, value))` stages a write.
- `.send(("delete", key))` stages a deletion.
- `.send(("commit",))` applies all staged changes and terminates.
- `.send(("rollback",))` discards changes and terminates.
- `.close()` must roll back, not commit.
- Invalid commands raise `ValueError` and roll back.
- Cleanup must be observable in an audit log.

This makes success explicit instead of treating `close()` as commit.

### Solution 12

In [19]:
Command = tuple[Any, ...]


@dataclass
class InMemoryDB:
    data: dict[str, Any] = field(default_factory=dict)


def transaction(
    db: InMemoryDB,
    audit: list[str],
) -> Generator[None, Command, None]:
    staged = dict(db.data)
    committed = False
    audit.append("begin")

    try:
        while True:
            command = yield
            if not command:
                raise ValueError("empty command")

            operation = command[0]

            if operation == "set" and len(command) == 3:
                _, key, value = command
                staged[str(key)] = value
                audit.append(f"stage:set:{key}")

            elif operation == "delete" and len(command) == 2:
                _, key = command
                staged.pop(str(key), None)
                audit.append(f"stage:delete:{key}")

            elif operation == "commit" and len(command) == 1:
                db.data = staged
                committed = True
                audit.append("commit")
                return

            elif operation == "rollback" and len(command) == 1:
                audit.append("rollback:explicit")
                return

            else:
                raise ValueError(f"invalid command: {command!r}")

    except GeneratorExit:
        audit.append("rollback:close")
        raise

    except Exception:
        audit.append("rollback:error")
        raise

    finally:
        audit.append("release")
        if not committed:
            # No mutation of db.data occurred, so staged changes vanish.
            pass


# Successful commit
db = InMemoryDB({"a": 1})
audit: list[str] = []
tx = transaction(db, audit)
next(tx)
tx.send(("set", "b", 2))
tx.send(("delete", "a"))

try:
    tx.send(("commit",))
except StopIteration:
    pass

assert db.data == {"b": 2}
assert audit == [
    "begin",
    "stage:set:b",
    "stage:delete:a",
    "commit",
    "release",
]

print(db.data, audit)

{'b': 2} ['begin', 'stage:set:b', 'stage:delete:a', 'commit', 'release']


### Rollback checks

In [20]:
# close() rolls back
db_close = InMemoryDB({"stable": 1})
audit_close: list[str] = []
tx_close = transaction(db_close, audit_close)
next(tx_close)
tx_close.send(("set", "temporary", 999))
tx_close.close()

assert db_close.data == {"stable": 1}
assert audit_close[-2:] == ["rollback:close", "release"]

# Explicit rollback
db_explicit = InMemoryDB({"stable": 1})
audit_explicit: list[str] = []
tx_explicit = transaction(db_explicit, audit_explicit)
next(tx_explicit)
tx_explicit.send(("set", "temporary", 999))
try:
    tx_explicit.send(("rollback",))
except StopIteration:
    pass

assert db_explicit.data == {"stable": 1}
assert audit_explicit[-2:] == ["rollback:explicit", "release"]

# Invalid command rolls back and propagates
db_error = InMemoryDB({"stable": 1})
audit_error: list[str] = []
tx_error = transaction(db_error, audit_error)
next(tx_error)

try:
    tx_error.send(("unknown",))
except ValueError as exc:
    assert "invalid command" in str(exc)

assert db_error.data == {"stable": 1}
assert audit_error[-2:] == ["rollback:error", "release"]

print("close   :", audit_close)
print("explicit:", audit_explicit)
print("error   :", audit_error)

close   : ['begin', 'stage:set:temporary', 'rollback:close', 'release']
explicit: ['begin', 'stage:set:temporary', 'rollback:explicit', 'release']
error   : ['begin', 'rollback:error', 'release']


### Best-practice conclusion

A transaction should normally commit only after an explicit success signal. Using `close()` as commit can accidentally commit partial work when a caller stops early.

# Part V — Testing and reusable abstractions

## Problem 13 — Build a close-aware iterator wrapper

Implement `CloseAwareIterator` around any iterator.

Requirements:

- Delegate `__next__`.
- Provide an idempotent `.close()`.
- Forward close to the wrapped iterator if supported.
- Track whether the wrapper itself is closed.
- After close, `next(wrapper)` must raise `StopIteration`.
- Support use as a context manager.

### Solution 13

In [21]:
class CloseAwareIterator(Iterator[T]):
    def __init__(self, source: Iterable[T]) -> None:
        self._iterator = iter(source)
        self._closed = False

    @property
    def closed(self) -> bool:
        return self._closed

    def __iter__(self) -> "CloseAwareIterator[T]":
        return self

    def __next__(self) -> T:
        if self._closed:
            raise StopIteration
        try:
            return next(self._iterator)
        except StopIteration:
            self.close()
            raise

    def close(self) -> None:
        if self._closed:
            return

        self._closed = True
        close = getattr(self._iterator, "close", None)
        if callable(close):
            close()

    def __enter__(self) -> "CloseAwareIterator[T]":
        return self

    def __exit__(self, exc_type, exc, tb) -> bool:
        self.close()
        return False


audit: list[str] = []
with CloseAwareIterator(auditable_source(audit)) as wrapped:
    assert next(wrapped) == 0
    assert next(wrapped) == 1
    assert not wrapped.closed

assert wrapped.closed
assert audit == ["source-open", "source-close"]

try:
    next(wrapped)
except StopIteration:
    pass
else:
    raise AssertionError("next() should fail after close")

wrapped.close()  # idempotent
print("CloseAwareIterator passed.")

CloseAwareIterator passed.


## Problem 14 — Create a cleanup test matrix

Write one generator and test all important termination paths:

1. close before start,
2. close after start,
3. normal exhaustion,
4. exception raised inside the generator,
5. application exception injected with `throw()`.

The audit must distinguish whether the body entered, whether cleanup ran, and what caused termination.

### Solution 14

In [22]:
class SourceFailure(RuntimeError):
    pass


def matrix_generator(
    audit: list[str],
    *,
    fail_internally: bool = False,
) -> Iterator[int]:
    audit.append("entered")
    reason = "normal"

    try:
        yield 1
        if fail_internally:
            reason = "internal-error"
            raise SourceFailure("boom")
        yield 2

    except GeneratorExit:
        reason = "close"
        raise

    except Exception as exc:
        if reason == "normal":
            reason = f"injected:{type(exc).__name__}"
        raise

    finally:
        audit.append(f"cleanup:{reason}")


results: dict[str, list[str]] = {}

# 1. Close before start
audit = []
g = matrix_generator(audit)
g.close()
results["before_start"] = audit
assert audit == []

# 2. Close after start
audit = []
g = matrix_generator(audit)
assert next(g) == 1
g.close()
results["after_start"] = audit
assert audit == ["entered", "cleanup:close"]

# 3. Normal exhaustion
audit = []
g = matrix_generator(audit)
assert list(g) == [1, 2]
results["exhaustion"] = audit
assert audit == ["entered", "cleanup:normal"]

# 4. Internal error
audit = []
g = matrix_generator(audit, fail_internally=True)
assert next(g) == 1
try:
    next(g)
except SourceFailure:
    pass
results["internal_error"] = audit
assert audit == ["entered", "cleanup:internal-error"]

# 5. Injected error
audit = []
g = matrix_generator(audit)
assert next(g) == 1
try:
    g.throw(KeyError("injected"))
except KeyError:
    pass
results["throw"] = audit
assert audit == ["entered", "cleanup:injected:KeyError"]

for name, events in results.items():
    print(f"{name:>14}: {events}")

  before_start: []
   after_start: ['entered', 'cleanup:close']
    exhaustion: ['entered', 'cleanup:normal']
internal_error: ['entered', 'cleanup:internal-error']
         throw: ['entered', 'cleanup:injected:KeyError']


# Part VI — Async generator shutdown

## Problem 15 — Close an async generator deterministically

Implement `async_counter(audit)` that:

- records open and close,
- yields increasing integers,
- awaits briefly between values,
- cleans up in `finally`.

Consume two values and call `aclose()`. Verify the audit and then show the equivalent `async with aclosing(...)` pattern when available.

### Solution 15

In [23]:
async def async_counter(audit: list[str]) -> AsyncIterator[int]:
    audit.append("async-open")
    value = 0
    try:
        while True:
            await asyncio.sleep(0)
            yield value
            value += 1
    finally:
        audit.append("async-close")


async def async_close_demo() -> list[str]:
    audit: list[str] = []
    gen = async_counter(audit)

    assert await anext(gen) == 0
    assert await anext(gen) == 1

    await gen.aclose()
    assert audit == ["async-open", "async-close"]
    return audit


async_audit = await async_close_demo()
print(async_audit)

['async-open', 'async-close']


### Solution 15B — `contextlib.aclosing`

`aclosing()` is the async counterpart of `closing()`. It is useful when an `async for` loop exits early.

In [24]:
from contextlib import aclosing


async def async_context_demo() -> tuple[list[int], list[str]]:
    audit: list[str] = []
    values: list[int] = []

    async with aclosing(async_counter(audit)) as gen:
        async for value in gen:
            values.append(value)
            if value == 2:
                break

    return values, audit


async_values, async_audit = await async_context_demo()
assert async_values == [0, 1, 2]
assert async_audit == ["async-open", "async-close"]

print(async_values, async_audit)

[0, 1, 2] ['async-open', 'async-close']


# Part VII — Capstone problem

## Problem 16 — Production-style CSV event stream

Implement `read_events_csv(path, audit, strict=True)`.

Input columns:

- `event_id`: non-empty string
- `user_id`: non-empty string
- `amount`: decimal-compatible number represented as text
- `active`: one of `true`, `false`, `1`, `0`, case-insensitive

Requirements:

1. Open lazily.
2. Use `csv.DictReader`.
3. Validate the exact header set.
4. Convert each row into an immutable `Event`.
5. In strict mode, raise `CSVValidationError` on the first invalid row.
6. In non-strict mode, append an error entry to `audit` and continue.
7. Always close the file.
8. Support safe early termination.
9. Avoid `eval()`.
10. Include tests for valid input, malformed rows, bad headers, and early close.

### Solution 16

In [25]:
from decimal import Decimal, InvalidOperation


@dataclass(frozen=True)
class Event:
    event_id: str
    user_id: str
    amount: Decimal
    active: bool


class CSVValidationError(ValueError):
    pass


EXPECTED_FIELDS = {"event_id", "user_id", "amount", "active"}


def parse_bool(text: str) -> bool:
    normalized = text.strip().lower()
    if normalized in {"true", "1"}:
        return True
    if normalized in {"false", "0"}:
        return False
    raise CSVValidationError(f"invalid boolean: {text!r}")


def parse_event(row: dict[str, str | None], line_number: int) -> Event:
    try:
        event_id = (row["event_id"] or "").strip()
        user_id = (row["user_id"] or "").strip()
        amount_text = (row["amount"] or "").strip()
        active_text = row["active"] or ""
    except KeyError as exc:
        raise CSVValidationError(
            f"line {line_number}: missing column {exc.args[0]!r}"
        ) from exc

    if not event_id:
        raise CSVValidationError(f"line {line_number}: empty event_id")
    if not user_id:
        raise CSVValidationError(f"line {line_number}: empty user_id")

    try:
        amount = Decimal(amount_text)
    except InvalidOperation as exc:
        raise CSVValidationError(
            f"line {line_number}: invalid amount {amount_text!r}"
        ) from exc

    try:
        active = parse_bool(active_text)
    except CSVValidationError as exc:
        raise CSVValidationError(f"line {line_number}: {exc}") from exc

    return Event(
        event_id=event_id,
        user_id=user_id,
        amount=amount,
        active=active,
    )


def read_events_csv(
    path: Path,
    audit: list[str],
    *,
    strict: bool = True,
) -> Iterator[Event]:
    audit.append("open")
    file = path.open("r", encoding="utf-8", newline="")

    try:
        reader = csv.DictReader(file)

        if reader.fieldnames is None:
            raise CSVValidationError("missing CSV header")

        actual_fields = {field.strip() for field in reader.fieldnames}
        if actual_fields != EXPECTED_FIELDS:
            raise CSVValidationError(
                f"unexpected headers: {sorted(actual_fields)!r}; "
                f"expected {sorted(EXPECTED_FIELDS)!r}"
            )

        for line_number, row in enumerate(reader, start=2):
            try:
                yield parse_event(row, line_number)
            except CSVValidationError as exc:
                if strict:
                    raise
                audit.append(f"skip:{exc}")

    finally:
        file.close()
        audit.append("close")

### Capstone tests — Valid input and early close

In [26]:
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "events.csv"
    path.write_text(
        "event_id,user_id,amount,active\n"
        "e1,u1,10.50,true\n"
        "e2,u2,-3.25,0\n"
        "e3,u1,7,1\n",
        encoding="utf-8",
    )

    # Full consumption
    audit_full: list[str] = []
    events = list(read_events_csv(path, audit_full))

    assert events == [
        Event("e1", "u1", Decimal("10.50"), True),
        Event("e2", "u2", Decimal("-3.25"), False),
        Event("e3", "u1", Decimal("7"), True),
    ]
    assert audit_full == ["open", "close"]

    # Early close
    audit_early: list[str] = []
    gen = read_events_csv(path, audit_early)
    assert next(gen).event_id == "e1"
    gen.close()
    assert audit_early == ["open", "close"]

print(events)
print("full :", audit_full)
print("early:", audit_early)

[Event(event_id='e1', user_id='u1', amount=Decimal('10.50'), active=True), Event(event_id='e2', user_id='u2', amount=Decimal('-3.25'), active=False), Event(event_id='e3', user_id='u1', amount=Decimal('7'), active=True)]
full : ['open', 'close']
early: ['open', 'close']


### Capstone tests — Strict and non-strict validation

In [27]:
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "events.csv"
    path.write_text(
        "event_id,user_id,amount,active\n"
        "e1,u1,10.50,true\n"
        "e2,u2,not-a-number,false\n"
        "e3,u3,5.00,yes\n"
        "e4,u4,8.00,1\n",
        encoding="utf-8",
    )

    # Strict: fail on first invalid row, then cleanup.
    strict_audit: list[str] = []
    strict_gen = read_events_csv(path, strict_audit, strict=True)

    assert next(strict_gen).event_id == "e1"

    try:
        next(strict_gen)
    except CSVValidationError as exc:
        assert "line 3" in str(exc)
        assert "invalid amount" in str(exc)

    assert strict_audit == ["open", "close"]
    assert inspect.getgeneratorstate(strict_gen) == "GEN_CLOSED"

    # Non-strict: skip invalid rows and continue.
    relaxed_audit: list[str] = []
    relaxed_events = list(
        read_events_csv(path, relaxed_audit, strict=False)
    )

    assert [event.event_id for event in relaxed_events] == ["e1", "e4"]
    assert relaxed_audit[0] == "open"
    assert relaxed_audit[-1] == "close"
    assert sum(item.startswith("skip:") for item in relaxed_audit) == 2

print("strict audit :", strict_audit)
print("relaxed rows :", relaxed_events)
print("relaxed audit:", relaxed_audit)

strict audit : ['open', 'close']
relaxed rows : [Event(event_id='e1', user_id='u1', amount=Decimal('10.50'), active=True), Event(event_id='e4', user_id='u4', amount=Decimal('8.00'), active=True)]
relaxed audit: ['open', "skip:line 3: invalid amount 'not-a-number'", "skip:line 4: invalid boolean: 'yes'", 'close']


### Capstone tests — Header validation

In [28]:
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "bad_headers.csv"
    path.write_text(
        "event_id,user,amount,active\n"
        "e1,u1,10,true\n",
        encoding="utf-8",
    )

    audit: list[str] = []
    gen = read_events_csv(path, audit)

    try:
        next(gen)
    except CSVValidationError as exc:
        assert "unexpected headers" in str(exc)

    assert audit == ["open", "close"]
    assert inspect.getgeneratorstate(gen) == "GEN_CLOSED"

print(audit)

['open', 'close']


### Capstone extension — Aggregate safely with early termination

Implement `sum_until_limit(path, limit)`:

- sum event amounts,
- stop before the sum would exceed `limit`,
- return `(total, accepted_event_ids)`,
- guarantee immediate generator cleanup.

In [29]:
def sum_until_limit(
    path: Path,
    limit: Decimal,
    audit: list[str],
) -> tuple[Decimal, list[str]]:
    total = Decimal("0")
    accepted: list[str] = []

    with closing(read_events_csv(path, audit)) as events:
        for event in events:
            candidate = total + event.amount
            if candidate > limit:
                break
            total = candidate
            accepted.append(event.event_id)

    return total, accepted


with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "events.csv"
    path.write_text(
        "event_id,user_id,amount,active\n"
        "e1,u1,4.50,true\n"
        "e2,u2,3.25,true\n"
        "e3,u3,9.00,true\n"
        "e4,u4,1.00,true\n",
        encoding="utf-8",
    )

    audit: list[str] = []
    total, accepted = sum_until_limit(path, Decimal("10"), audit)

    assert total == Decimal("7.75")
    assert accepted == ["e1", "e2"]
    assert audit == ["open", "close"]

print(total, accepted, audit)

7.75 ['e1', 'e2'] ['open', 'close']


# Part VIII — Additional advanced drills

## Drill A — Returning a value on normal exhaustion

A generator may return a value, which becomes `StopIteration.value` on normal exhaustion. It is not a value yielded during `close()`.

Predict and verify the result below.

In [30]:
def generator_with_result() -> Iterator[int]:
    yield 1
    yield 2
    return 99


gen = generator_with_result()
assert next(gen) == 1
assert next(gen) == 2

try:
    next(gen)
except StopIteration as exc:
    assert exc.value == 99
    print("Returned value:", exc.value)

Returned value: 99


## Drill B — A generator that raises another exception during close

If a generator raises an exception other than `GeneratorExit` while processing `close()`, that exception propagates to the caller.

In [31]:
class CleanupFailure(RuntimeError):
    pass


def failing_cleanup() -> Iterator[str]:
    try:
        yield "resource"
    except GeneratorExit:
        raise CleanupFailure("cleanup failed")


gen = failing_cleanup()
assert next(gen) == "resource"

try:
    gen.close()
except CleanupFailure as exc:
    assert str(exc) == "cleanup failed"
    print("Propagated:", exc)

Propagated: cleanup failed


## Drill C — Avoid suppressing cleanup failures accidentally

A `return` statement inside `finally` suppresses active exceptions. This is usually a serious bug in resource-management code.

In [32]:
def bad_finally() -> Iterator[int]:
    try:
        yield 1
        raise ValueError("important failure")
    finally:
        return  # suppresses ValueError by terminating the generator


gen = bad_finally()
assert next(gen) == 1

try:
    next(gen)
except StopIteration:
    print("The ValueError was suppressed — this is why return-in-finally is risky.")

The ValueError was suppressed — this is why return-in-finally is risky.


## Drill D — Close a chain of wrappers

Each wrapper should close its immediate upstream iterator. Verify cleanup occurs exactly once.

In [33]:
def filter_closing(predicate, source: Iterable[T]) -> Iterator[T]:
    iterator = iter(source)
    close = getattr(iterator, "close", None)
    try:
        for item in iterator:
            if predicate(item):
                yield item
    finally:
        if callable(close):
            close()


audit: list[str] = []
source = auditable_source(audit)
mapped = map_closing(lambda x: x + 1, source)
filtered = filter_closing(lambda x: x % 2 == 0, mapped)

assert next(filtered) == 2
assert next(filtered) == 4

filtered.close()
assert audit == ["source-open", "source-close"]

print(audit)

['source-open', 'source-close']


# Review checklist

Use this checklist when reviewing generator-based resource code:

- Does the generator acquire the resource inside its body, not at generator-object creation time?
- Is cleanup in `finally`?
- Is the owner explicit about early termination?
- Is `.close()` called deterministically?
- Does code avoid yielding after `GeneratorExit`?
- Are application exceptions allowed to propagate unless there is a documented recovery policy?
- Does a wrapper close its upstream generator when it owns it?
- Are transaction commit and rollback semantics explicit?
- Are cleanup actions idempotent?
- Are tests checking normal exhaustion, early close, and error paths?
- Is async cleanup handled with `aclose()` or `aclosing()`?
- Is garbage collection excluded from correctness assumptions?

# Final challenge prompts

These are intentionally left without additional solutions because the notebook already contains all required techniques:

1. Add line-number and source-file metadata to every `Event`.
2. Build a `merge_sorted_closing(*sources, key=...)` generator that closes all unfinished sources.
3. Implement a timeout-aware async generator consumer that calls `aclose()` after cancellation.
4. Write a decorator that automatically primes a generator coroutine, and document how it should handle immediate termination.
5. Add property-based tests that generate malformed CSV rows and prove the file always closes.
6. Replace the audit lists with structured event objects and assert exact cleanup ordering.

# Summary

The central rule is simple:

> A generator that owns a resource must release it in `finally`, and the consumer that stops early must close it deterministically.

`GeneratorExit` is a shutdown signal. A generator may re-raise it or terminate normally, but must not yield another value in response. `yield from`, `closing()`, `ExitStack`, wrapper iterators, explicit transaction protocols, and `aclosing()` are practical tools for building robust systems around that rule.